# Strong Model Zoo v3

Блендим (не очень успешно)

In [1]:
# =========================
# 0. Config
# =========================
from pathlib import Path
import os
import gc
import json
import time
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    HAS_STRATIFIED_GROUP_KFOLD = False

from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from scipy.stats import rankdata

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

RANDOM_STATE = 42
N_JOBS = max(1, (os.cpu_count() or 8) - 1)

CV_N_SPLITS = 5
CV_SEED = 42

# Main runtime switches.
RUN_LIGHTGBM = True
RUN_SKLEARN_MODELS = True
RUN_XGBOOST = True
RUN_CATBOOST = True
RUN_FLAML = True
RUN_MLP = True
RUN_RANDOM_FORESTS = True

# AutoML budget. Keep sane for local overnight run.
FLAML_TIME_BUDGET_PER_FOLD = 20 * 60

# Blend settings.
MIN_MODEL_AUC_FOR_BLEND = 0.60
TOP_K_FOR_SIMPLE_BLEND = 12
RANDOM_WEIGHT_SEARCH_TRIALS = 3500
RANDOM_WEIGHT_SEARCH_META_FOLDS = 5

OUTPUT_SUBDIR = 'submissions_v3'
OUTPUT_MODEL_DIR = 'model_outputs_v3'
DATA_DIR_NAME = 'prepared_data_v3'

In [2]:
# =========================
# 1. Paths and data loading
# =========================
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / DATA_DIR_NAME).exists() or (p / 'prepared_data').exists() or (p / 'train.csv').exists() or (p/'data/train.csv').exists():
            return p
    return start

ROOT = find_repo_root(Path.cwd())
DATA_DIR = ROOT / DATA_DIR_NAME
if not DATA_DIR.exists():
    raise FileNotFoundError(f'{DATA_DIR} does not exist. Run feature_space_select_best_v3.ipynb first.')

SUB_DIR = ROOT / OUTPUT_SUBDIR
MODEL_DIR = ROOT / OUTPUT_MODEL_DIR
SUB_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)
print('SUB_DIR =', SUB_DIR)

X = pd.read_parquet(DATA_DIR/'X_train_best_dataset_v3.parquet')
X_test = pd.read_parquet(DATA_DIR/'X_test_best_dataset_v3.parquet')
y_df = pd.read_csv(DATA_DIR/'y_train.csv')
y = y_df['target'].astype(int).values
train_index = pd.read_csv(DATA_DIR/'train_index.csv')['index']
test_index = pd.read_csv(DATA_DIR/'test_index.csv')['index']

group_path = DATA_DIR/'group_id_v3.csv'
if group_path.exists():
    groups = pd.read_csv(group_path)['group_id'].astype(str).values
else:
    print('WARNING: group_id_v3.csv not found; using row index as group.')
    groups = np.arange(len(X)).astype(str)

sample_sub_path = DATA_DIR/'sample_submission.csv'
if sample_sub_path.exists():
    sample_submission = pd.read_csv(sample_sub_path)
else:
    sample_submission = pd.DataFrame({'index': test_index, 'score': 0.0})

with open(DATA_DIR/'best_dataset_v3_info.json') as f:
    best_info = json.load(f)

print('Best dataset:', best_info.get('best_name'))
print('X:', X.shape, 'X_test:', X_test.shape, 'target mean:', y.mean(), 'positives:', int(y.sum()))
print('Columns unique:', X.columns.is_unique, 'aligned:', list(X.columns)==list(X_test.columns))

display(pd.DataFrame([best_info.get('stage2_result', {})]))

ROOT = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton
DATA_DIR = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data_v3
SUB_DIR = /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/submissions_v3


Best dataset: raw_all+row+block+hash+freq_enc+svd_pca+mask_svd+kmeans_proj
X: (247972, 2338) X_test: (106274, 2338) target mean: 0.013493458938912458 positives: 3346
Columns unique: True aligned: True


,name,raw_set,blocks,n_features,mean_auc,std_auc,min_auc,max_auc,n_folds_total,elapsed_sec,selection_score
0,raw_all+row+block+hash+freq_enc+svd_pca+mask_s...,raw_all,row_stats+block_stats+hash_features+freq_enc+s...,2338,0.644796,0.00826,0.633348,0.659808,10,312.445279,0.642394


In [3]:
# =========================
# 2. Utilities
# =========================
def make_cv_splits(y, groups, n_splits=5, seed=42):
    if HAS_STRATIFIED_GROUP_KFOLD:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(cv.split(np.zeros(len(y)), y, groups=groups))
    print('WARNING: StratifiedGroupKFold unavailable. Falling back to StratifiedKFold.')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(cv.split(np.zeros(len(y)), y))

splits = make_cv_splits(y, groups, CV_N_SPLITS, CV_SEED)
for i, (tr, va) in enumerate(splits):
    overlap = len(set(groups[tr]).intersection(set(groups[va])))
    print(f'fold {i}: train={len(tr)}, valid={len(va)}, valid_pos={int(y[va].sum())}, group_overlap={overlap}')
    assert overlap == 0, 'Group leakage detected between train and valid'

model_oof = {}
model_test = {}
model_results = []
fold_results = []

# Common finite cleaning. Tree models can handle NaN, linear models use imputers in pipeline.
X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Some libraries dislike duplicate column names or non-string names.
X.columns = X.columns.astype(str)
X_test.columns = X_test.columns.astype(str)


def make_submission(pred, name):
    pred = np.asarray(pred, dtype=float)
    pred = np.clip(pred, 0, 1)
    sub = sample_submission.copy()
    if 'score' in sub.columns:
        sub['score'] = pred
    else:
        # fallback: last column is usually target score
        sub[sub.columns[-1]] = pred
    # Enforce suffix _v3.
    if not name.endswith('_v3'):
        name = name + '_v3'
    path = SUB_DIR / f'submission_{name}.csv'
    sub.to_csv(path, index=False)
    print('saved', path.name)
    return path


def save_predictions(name, oof, test_pred):
    if not name.endswith('_v3'):
        out_name = name + '_v3'
    else:
        out_name = name
    pd.DataFrame({'index': train_index, 'target': y, 'oof_pred': oof}).to_csv(MODEL_DIR/f'oof_{out_name}.csv', index=False)
    pd.DataFrame({'index': test_index, 'score': test_pred}).to_csv(MODEL_DIR/f'test_pred_{out_name}.csv', index=False)
    make_submission(test_pred, out_name)


def add_model_result(name, oof, test_pred, extra=None):
    auc = roc_auc_score(y, oof)
    row = {'model': name, 'oof_auc': auc, 'test_pred_mean': float(np.mean(test_pred)), 'test_pred_std': float(np.std(test_pred))}
    if extra:
        row.update(extra)
    model_results.append(row)
    model_oof[name] = np.asarray(oof)
    model_test[name] = np.asarray(test_pred)
    save_predictions(name, oof, test_pred)
    print(f'{name}: OOF AUC={auc:.6f}')
    pd.DataFrame(model_results).sort_values('oof_auc', ascending=False).to_csv(MODEL_DIR/'model_results_v3.csv', index=False)
    return auc

neg_pos_ratio = max(1.0, (len(y) - y.sum()) / max(1, y.sum()))
print('scale_pos_weight / neg_pos_ratio:', neg_pos_ratio)

fold 0: train=198376, valid=49596, valid_pos=670, group_overlap=0
fold 1: train=198378, valid=49594, valid_pos=669, group_overlap=0
fold 2: train=198378, valid=49594, valid_pos=669, group_overlap=0


fold 3: train=198378, valid=49594, valid_pos=669, group_overlap=0
fold 4: train=198378, valid=49594, valid_pos=669, group_overlap=0


scale_pos_weight / neg_pos_ratio: 73.10998206814106


In [4]:
# =========================
# 3. LightGBM model zoo
# =========================
try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print('LightGBM unavailable:', repr(e))


def run_lgbm_model(name, params):
    if not HAS_LGBM or not RUN_LIGHTGBM:
        print('skip', name)
        return None
    oof = np.zeros(len(X), dtype='float32')
    test_pred = np.zeros(len(X_test), dtype='float64')
    best_iters = []
    t0 = time.time()
    for fold, (tr_idx, va_idx) in enumerate(splits):
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]
        spw = max(1.0, (len(ytr)-ytr.sum()) / max(1, ytr.sum()))
        p = deepcopy(params)
        p.setdefault('objective', 'binary')
        p.setdefault('metric', 'auc')
        p.setdefault('n_jobs', N_JOBS)
        p.setdefault('verbosity', -1)
        p.setdefault('random_state', RANDOM_STATE + fold)
        p.setdefault('scale_pos_weight', spw)
        model = lgb.LGBMClassifier(**p)
        callbacks = [lgb.log_evaluation(0)]
        if p.get('boosting_type', 'gbdt') != 'dart':
            callbacks.append(lgb.early_stopping(120, verbose=False))
        model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='auc', callbacks=callbacks)
        pred_va = model.predict_proba(Xva)[:, 1]
        pred_te = model.predict_proba(X_test)[:, 1]
        oof[va_idx] = pred_va
        test_pred += pred_te / len(splits)
        auc = roc_auc_score(yva, pred_va)
        best_iter = getattr(model, 'best_iteration_', None)
        best_iters.append(best_iter if best_iter else p.get('n_estimators'))
        fold_results.append({'model': name, 'fold': fold, 'auc': auc, 'best_iteration': best_iter})
        print(f'{name} fold {fold}: AUC={auc:.6f}, best_iter={best_iter}')
        del model, Xtr, Xva, pred_va, pred_te; gc.collect()
    extra = {'elapsed_sec': time.time()-t0, 'mean_best_iter': float(np.nanmean(best_iters))}
    return add_model_result(name, oof, test_pred, extra)

lgb_configs = {
    'lgb_regularized': dict(
        n_estimators=3500, learning_rate=0.018, num_leaves=48, min_child_samples=90,
        subsample=0.82, subsample_freq=1, colsample_bytree=0.78,
        reg_alpha=0.25, reg_lambda=35.0, extra_trees=True,
    ),
    'lgb_deeper': dict(
        n_estimators=3000, learning_rate=0.015, num_leaves=96, min_child_samples=45,
        subsample=0.78, subsample_freq=1, colsample_bytree=0.72,
        reg_alpha=0.10, reg_lambda=25.0,
    ),
    'lgb_small_leaves': dict(
        n_estimators=2800, learning_rate=0.022, num_leaves=24, min_child_samples=140,
        subsample=0.90, subsample_freq=1, colsample_bytree=0.88,
        reg_alpha=0.50, reg_lambda=55.0, extra_trees=True,
    ),
    'lgb_goss': dict(
        boosting_type='goss', n_estimators=2600, learning_rate=0.018, num_leaves=56,
        min_child_samples=70, colsample_bytree=0.80, reg_alpha=0.2, reg_lambda=30.0,
    ),
    'lgb_rf': dict(
        boosting_type='rf', n_estimators=1200, learning_rate=0.05, num_leaves=64,
        min_child_samples=50, subsample=0.75, subsample_freq=1, colsample_bytree=0.75,
        reg_alpha=0.1, reg_lambda=20.0,
    ),
    'lgb_dart': dict(
        boosting_type='dart', n_estimators=1200, learning_rate=0.025, num_leaves=48,
        min_child_samples=80, subsample=0.85, subsample_freq=1, colsample_bytree=0.75,
        reg_alpha=0.1, reg_lambda=25.0, drop_rate=0.08, skip_drop=0.55,
    ),
}

for name, params in lgb_configs.items():
    run_lgbm_model(name, params)

pd.DataFrame(fold_results).to_csv(MODEL_DIR/'fold_results_v3.csv', index=False)

lgb_regularized fold 0: AUC=0.645558, best_iter=341


lgb_regularized fold 1: AUC=0.645568, best_iter=339


lgb_regularized fold 2: AUC=0.652173, best_iter=436


lgb_regularized fold 3: AUC=0.634696, best_iter=406


lgb_regularized fold 4: AUC=0.646161, best_iter=281


...

lgb_dart fold 1: AUC=0.646469, best_iter=0


lgb_dart fold 2: AUC=0.638116, best_iter=0


lgb_dart fold 3: AUC=0.630743, best_iter=0


lgb_dart fold 4: AUC=0.636566, best_iter=0


saved submission_lgb_dart_v3.csv
lgb_dart: OOF AUC=0.638402


In [6]:
# =========================
# 5. XGBoost / CatBoost / FLAML AutoML
# =========================
if RUN_XGBOOST:
    try:
        from xgboost import XGBClassifier
        def run_xgb_model(name, params):
            oof = np.zeros(len(X), dtype='float32')
            test_pred = np.zeros(len(X_test), dtype='float64')
            t0 = time.time()
            for fold, (tr_idx, va_idx) in enumerate(splits):
                ytr, yva = y[tr_idx], y[va_idx]
                spw = max(1.0, (len(ytr)-ytr.sum())/max(1, ytr.sum()))
                p = deepcopy(params)
                p.setdefault('scale_pos_weight', spw)
                p.setdefault('tree_method', 'hist')
                p.setdefault('eval_metric', 'auc')
                p.setdefault('n_jobs', N_JOBS)
                p.setdefault('random_state', RANDOM_STATE+fold)
                model = XGBClassifier(**p)
                model.fit(
                    X.iloc[tr_idx].fillna(0), ytr,
                    eval_set=[(X.iloc[va_idx].fillna(0), yva)],
                    verbose=False,
                )
                pred_va = model.predict_proba(X.iloc[va_idx].fillna(0))[:, 1]
                pred_te = model.predict_proba(X_test.fillna(0))[:, 1]
                oof[va_idx] = pred_va
                test_pred += pred_te / len(splits)
                auc = roc_auc_score(yva, pred_va)
                fold_results.append({'model': name, 'fold': fold, 'auc': auc, 'best_iteration': getattr(model, 'best_iteration', None)})
                print(f'{name} fold {fold}: AUC={auc:.6f}')
                del model, pred_va, pred_te; gc.collect()
            return add_model_result(name, oof, test_pred, {'elapsed_sec': time.time()-t0})

        run_xgb_model('xgb_depth4', dict(
            n_estimators=1600, learning_rate=0.025, max_depth=4, min_child_weight=8,
            subsample=0.82, colsample_bytree=0.78, reg_alpha=0.2, reg_lambda=12.0,
            max_bin=256,
        ))
        run_xgb_model('xgb_depth6_regularized', dict(
            n_estimators=1300, learning_rate=0.02, max_depth=6, min_child_weight=12,
            subsample=0.78, colsample_bytree=0.70, reg_alpha=0.4, reg_lambda=25.0,
            max_bin=256,
        ))
    except Exception as e:
        print('XGBoost skipped:', repr(e))

if RUN_CATBOOST:
    try:
        from catboost import CatBoostClassifier
        def run_cat_model(name, params):
            oof = np.zeros(len(X), dtype='float32')
            test_pred = np.zeros(len(X_test), dtype='float64')
            t0 = time.time()
            for fold, (tr_idx, va_idx) in enumerate(splits):
                p = deepcopy(params)
                p.setdefault('loss_function', 'Logloss')
                p.setdefault('eval_metric', 'AUC')
                p.setdefault('random_seed', RANDOM_STATE+fold)
                p.setdefault('thread_count', N_JOBS)
                p.setdefault('allow_writing_files', False)
                p.setdefault('auto_class_weights', 'Balanced')
                model = CatBoostClassifier(**p)
                model.fit(
                    X.iloc[tr_idx], y[tr_idx],
                    eval_set=(X.iloc[va_idx], y[va_idx]),
                    verbose=False,
                    use_best_model=True,
                )
                pred_va = model.predict_proba(X.iloc[va_idx])[:, 1]
                pred_te = model.predict_proba(X_test)[:, 1]
                oof[va_idx] = pred_va
                test_pred += pred_te / len(splits)
                auc = roc_auc_score(y[va_idx], pred_va)
                fold_results.append({'model': name, 'fold': fold, 'auc': auc, 'best_iteration': model.get_best_iteration()})
                print(f'{name} fold {fold}: AUC={auc:.6f}')
                del model, pred_va, pred_te; gc.collect()
            return add_model_result(name, oof, test_pred, {'elapsed_sec': time.time()-t0})

        run_cat_model('catboost_depth6', dict(iterations=2200, learning_rate=0.025, depth=6, l2_leaf_reg=12.0, random_strength=0.8, od_type='Iter', od_wait=160))
        run_cat_model('catboost_depth4_ordered', dict(iterations=2500, learning_rate=0.022, depth=4, l2_leaf_reg=18.0, bootstrap_type='Bayesian', bagging_temperature=0.7, od_type='Iter', od_wait=160))
    except Exception as e:
        print('CatBoost skipped:', repr(e))

if RUN_FLAML:
    try:
        from flaml import AutoML
        def run_flaml_model(name):
            oof = np.zeros(len(X), dtype='float32')
            test_pred = np.zeros(len(X_test), dtype='float64')
            t0 = time.time()
            for fold, (tr_idx, va_idx) in enumerate(splits):
                automl = AutoML()
                settings = {
                    'time_budget': FLAML_TIME_BUDGET_PER_FOLD,
                    'metric': 'roc_auc',
                    'task': 'classification',
                    'estimator_list': ['lgbm', 'xgboost', 'catboost', 'rf', 'extra_tree'],
                    'eval_method': 'holdout',
                    'log_file_name': str(MODEL_DIR/f'flaml_{name}_fold{fold}.log'),
                    'seed': RANDOM_STATE + fold,
                    'n_jobs': N_JOBS,
                    'verbose': 0,
                }
                automl.fit(X_train=X.iloc[tr_idx], y_train=y[tr_idx], X_val=X.iloc[va_idx], y_val=y[va_idx], **settings)
                pred_va = automl.predict_proba(X.iloc[va_idx])[:, 1]
                pred_te = automl.predict_proba(X_test)[:, 1]
                oof[va_idx] = pred_va
                test_pred += pred_te / len(splits)
                auc = roc_auc_score(y[va_idx], pred_va)
                fold_results.append({'model': name, 'fold': fold, 'auc': auc, 'best_iteration': None, 'best_estimator': str(automl.best_estimator)})
                print(f'{name} fold {fold}: AUC={auc:.6f}, best_estimator={automl.best_estimator}')
                del automl, pred_va, pred_te; gc.collect()
            return add_model_result(name, oof, test_pred, {'elapsed_sec': time.time()-t0})
        run_flaml_model('flaml_automl')
    except Exception as e:
        print('FLAML skipped:', repr(e))

pd.DataFrame(fold_results).to_csv(MODEL_DIR/'fold_results_v3.csv', index=False)
pd.DataFrame(model_results).sort_values('oof_auc', ascending=False).to_csv(MODEL_DIR/'model_results_v3.csv', index=False)
display(pd.DataFrame(model_results).sort_values('oof_auc', ascending=False))

xgb_depth4 fold 0: AUC=0.638564


xgb_depth4 fold 1: AUC=0.633509


xgb_depth4 fold 2: AUC=0.634675


xgb_depth4 fold 3: AUC=0.624511


xgb_depth4 fold 4: AUC=0.625028


...

flaml_automl fold 2: AUC=0.664086, best_estimator=lgbm


flaml_automl fold 3: AUC=0.644037, best_estimator=lgbm


flaml_automl fold 4: AUC=0.655274, best_estimator=lgbm


saved submission_flaml_automl_v3.csv
flaml_automl: OOF AUC=0.655824


,model,oof_auc,test_pred_mean,test_pred_std,elapsed_sec,mean_best_iter
17,flaml_automl,0.655824,0.015429,0.008104,6012.592753,NaN
16,catboost_depth4_ordered,0.648936,0.450481,0.112065,587.753351,NaN
2,lgb_small_leaves,0.647175,0.398239,0.134869,146.279542,405.8
0,lgb_regularized,0.643600,0.373386,0.134538,162.606043,360.6
7,logreg_saga_l1_sparse,0.641800,0.448931,0.154380,3206.059304,NaN
6,logreg_saga_l2,0.641793,0.448940,0.154350,1872.403902,NaN
15,catboost_depth6,0.641726,0.431195,0.118448,421.771670,NaN
3,lgb_goss,0.639295,0.320512,0.140195,191.239505,304.8
5,lgb_dart,0.638402,0.282832,0.128561,858.638413,1200.0
9,hist_gradient_boosting,0.632863,0.436696,0.107855,609.986326,NaN


In [7]:
# =========================
# 6. Blending and stacking
# =========================
results_df = pd.DataFrame(model_results).sort_values('oof_auc', ascending=False).reset_index(drop=True)
results_df.to_csv(MODEL_DIR/'model_results_v3.csv', index=False)
display(results_df)

eligible = results_df[results_df['oof_auc'] >= MIN_MODEL_AUC_FOR_BLEND]['model'].tolist()
print('Eligible for blend:', eligible)
if len(eligible) < 2:
    raise RuntimeError('Need at least two eligible models for blending.')

def rank_normalize(pred):
    r = rankdata(pred, method='average')
    return (r - 1) / (len(r) - 1 + 1e-12)

def save_blend(name, oof_pred, test_pred, weights=None):
    auc = roc_auc_score(y, oof_pred)
    row = {'blend': name, 'oof_auc': auc, 'n_models': len(eligible)}
    blend_results.append(row)
    pd.DataFrame({'index': train_index, 'target': y, 'oof_pred': oof_pred}).to_csv(MODEL_DIR/f'oof_{name}_v3.csv', index=False)
    pd.DataFrame({'index': test_index, 'score': test_pred}).to_csv(MODEL_DIR/f'test_pred_{name}_v3.csv', index=False)
    if weights is not None:
        pd.DataFrame({'model': list(weights.keys()), 'weight': list(weights.values())}).to_csv(MODEL_DIR/f'weights_{name}_v3.csv', index=False)
    make_submission(test_pred, f'{name}_v3')
    print(f'{name}: OOF AUC={auc:.6f}')
    return auc

blend_results = []

# Top-K lists by OOF AUC.
top_models = eligible[:min(TOP_K_FOR_SIMPLE_BLEND, len(eligible))]
print('Top models for simple blends:', top_models)

# Equal probability blend.
oof_eq = np.mean([model_oof[m] for m in top_models], axis=0)
te_eq = np.mean([model_test[m] for m in top_models], axis=0)
save_blend('blend_equal_probability_topk', oof_eq, te_eq, {m: 1/len(top_models) for m in top_models})

# Equal rank blend.
oof_rank = np.mean([rank_normalize(model_oof[m]) for m in top_models], axis=0)
te_rank = np.mean([rank_normalize(model_test[m]) for m in top_models], axis=0)
save_blend('blend_equal_rank_topk', oof_rank, te_rank, {m: 1/len(top_models) for m in top_models})

# AUC-weighted probability blend.
auc_scores = np.array([max(0, results_df.set_index('model').loc[m, 'oof_auc'] - 0.5) for m in top_models], dtype=float)
w = auc_scores ** 2
w = w / (w.sum() + 1e-12)
oof_aucw = np.sum([w[i]*model_oof[m] for i, m in enumerate(top_models)], axis=0)
te_aucw = np.sum([w[i]*model_test[m] for i, m in enumerate(top_models)], axis=0)
save_blend('blend_auc_weighted_topk', oof_aucw, te_aucw, dict(zip(top_models, w)))

# Greedy forward blending with OOF. Aggressive but often useful.
def greedy_blend(models, max_steps=20):
    selected = []
    current_oof = None
    current_test = None
    best_auc = -1
    remaining = list(models)
    weights_trace = []
    while remaining and len(selected) < max_steps:
        best_candidate = None
        best_candidate_auc = best_auc
        best_candidate_oof = None
        best_candidate_test = None
        for m in remaining:
            if current_oof is None:
                cand_oof = model_oof[m]
                cand_test = model_test[m]
            else:
                # Try convex additions with a few weights.
                for alpha in [0.10, 0.15, 0.20, 0.25, 0.33, 0.50]:
                    tmp_oof = (1-alpha)*current_oof + alpha*model_oof[m]
                    tmp_auc = roc_auc_score(y, tmp_oof)
                    if tmp_auc > best_candidate_auc:
                        best_candidate_auc = tmp_auc
                        best_candidate = (m, alpha)
                        best_candidate_oof = tmp_oof
                        best_candidate_test = (1-alpha)*current_test + alpha*model_test[m]
                continue
            cand_auc = roc_auc_score(y, cand_oof)
            if cand_auc > best_candidate_auc:
                best_candidate_auc = cand_auc
                best_candidate = (m, 1.0)
                best_candidate_oof = cand_oof.copy()
                best_candidate_test = cand_test.copy()
        if best_candidate is None or best_candidate_auc <= best_auc + 1e-6:
            break
        m, alpha = best_candidate
        selected.append((m, alpha))
        remaining.remove(m)
        current_oof = best_candidate_oof
        current_test = best_candidate_test
        best_auc = best_candidate_auc
        weights_trace.append({'step': len(selected), 'model': m, 'alpha': alpha, 'auc': best_auc})
        print('greedy step', len(selected), m, alpha, best_auc)
    return current_oof, current_test, pd.DataFrame(weights_trace)

greedy_oof, greedy_test, greedy_trace = greedy_blend(top_models, max_steps=len(top_models))
greedy_trace.to_csv(MODEL_DIR/'greedy_blend_trace_v3.csv', index=False)
save_blend('blend_greedy_oof', greedy_oof, greedy_test)

,model,oof_auc,test_pred_mean,test_pred_std,elapsed_sec,mean_best_iter
0,flaml_automl,0.655824,0.015429,0.008104,6012.592753,NaN
1,catboost_depth4_ordered,0.648936,0.450481,0.112065,587.753351,NaN
2,lgb_small_leaves,0.647175,0.398239,0.134869,146.279542,405.8
3,lgb_regularized,0.643600,0.373386,0.134538,162.606043,360.6
4,logreg_saga_l1_sparse,0.641800,0.448931,0.154380,3206.059304,NaN
5,logreg_saga_l2,0.641793,0.448940,0.154350,1872.403902,NaN
6,catboost_depth6,0.641726,0.431195,0.118448,421.771670,NaN
7,lgb_goss,0.639295,0.320512,0.140195,191.239505,304.8
8,lgb_dart,0.638402,0.282832,0.128561,858.638413,1200.0
9,hist_gradient_boosting,0.632863,0.436696,0.107855,609.986326,NaN


Eligible for blend: ['flaml_automl', 'catboost_depth4_ordered', 'lgb_small_leaves', 'lgb_regularized', 'logreg_saga_l1_sparse', 'logreg_saga_l2', 'catboost_depth6', 'lgb_goss', 'lgb_dart', 'hist_gradient_boosting', 'lgb_deeper', 'xgb_depth4', 'xgb_depth6_regularized', 'extra_trees', 'random_forest_balanced']
Top models for simple blends: ['flaml_automl', 'catboost_depth4_ordered', 'lgb_small_leaves', 'lgb_regularized', 'logreg_saga_l1_sparse', 'logreg_saga_l2', 'catboost_depth6', 'lgb_goss', 'lgb_dart', 'hist_gradient_boosting', 'lgb_deeper', 'xgb_depth4']


saved submission_blend_equal_probability_topk_v3.csv
blend_equal_probability_topk: OOF AUC=0.652828


saved submission_blend_equal_rank_topk_v3.csv
blend_equal_rank_topk: OOF AUC=0.653613


saved submission_blend_auc_weighted_topk_v3.csv
blend_auc_weighted_topk: OOF AUC=0.653176


...

greedy step 2 catboost_depth4_ordered 0.1 0.6562172785381652


greedy step 3 logreg_saga_l2 0.1 0.6568357055384482


greedy step 4 lgb_deeper 0.1 0.6572683676694379


saved submission_blend_greedy_oof_v3.csv
blend_greedy_oof: OOF AUC=0.657268


0.6572683676694379

In [8]:
# CV weight search on OOF meta-folds.
# This is safer than optimizing weights once on all OOF, because it checks whether weights generalize across train rows.
def cv_weight_search(models, n_trials=3000, n_meta_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    P = np.column_stack([model_oof[m] for m in models])
    T = np.column_stack([model_test[m] for m in models])
    meta_splits = make_cv_splits(y, groups, n_meta_folds, seed+777)
    fold_weights = []
    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(meta_splits):
        best_w = None
        best_train_auc = -1
        # Include deterministic starts.
        starts = []
        starts.append(np.ones(len(models)) / len(models))
        auc_base = np.array([max(0, roc_auc_score(y[tr_idx], P[tr_idx, i]) - 0.5) for i in range(len(models))])
        starts.append((auc_base**2) / ((auc_base**2).sum() + 1e-12))
        for w0 in starts:
            auc0 = roc_auc_score(y[tr_idx], P[tr_idx] @ w0)
            if auc0 > best_train_auc:
                best_train_auc = auc0
                best_w = w0
        # Random Dirichlet search with different sparsity concentrations.
        for t in range(n_trials):
            alpha = rng.choice([0.15, 0.3, 0.6, 1.0, 2.0])
            w = rng.dirichlet(np.ones(len(models)) * alpha)
            # Occasional top-k sparse weights.
            if rng.random() < 0.35:
                k = int(rng.integers(2, min(len(models), 8)+1))
                keep = rng.choice(len(models), size=k, replace=False)
                w2 = np.zeros(len(models))
                w2[keep] = rng.dirichlet(np.ones(k) * alpha)
                w = w2
            pred_tr = P[tr_idx] @ w
            auc = roc_auc_score(y[tr_idx], pred_tr)
            # Slight regularization toward simpler/non-extreme weights.
            auc_reg = auc - 0.0004 * np.sum(w*w)
            if auc_reg > best_train_auc:
                best_train_auc = auc_reg
                best_w = w
        pred_va = P[va_idx] @ best_w
        va_auc = roc_auc_score(y[va_idx], pred_va)
        fold_weights.append(best_w)
        fold_aucs.append(va_auc)
        print(f'weight-search meta fold {fold}: valid AUC={va_auc:.6f}, train_objective={best_train_auc:.6f}')
    avg_w = np.mean(fold_weights, axis=0)
    avg_w = avg_w / (avg_w.sum() + 1e-12)
    oof_blend = P @ avg_w
    test_blend = T @ avg_w
    weights = dict(zip(models, avg_w))
    return oof_blend, test_blend, weights, fold_aucs

# Use top models only; too many weak/noisy models make random search unstable.
weight_models = top_models[:min(10, len(top_models))]
ws_oof, ws_test, ws_weights, ws_fold_aucs = cv_weight_search(
    weight_models,
    n_trials=RANDOM_WEIGHT_SEARCH_TRIALS,
    n_meta_folds=RANDOM_WEIGHT_SEARCH_META_FOLDS,
    seed=RANDOM_STATE,
)
pd.DataFrame({'meta_fold_auc': ws_fold_aucs}).to_csv(MODEL_DIR/'weight_search_meta_fold_aucs_v3.csv', index=False)
save_blend('blend_cv_weight_search', ws_oof, ws_test, ws_weights)

# Logistic stacker on OOF predictions. This is a simple meta-model; CV below is used as a sanity estimate.
def logistic_stacker(models):
    P = np.column_stack([model_oof[m] for m in models])
    T = np.column_stack([model_test[m] for m in models])
    P_rank = np.column_stack([rank_normalize(model_oof[m]) for m in models])
    T_rank = np.column_stack([rank_normalize(model_test[m]) for m in models])
    meta_X = np.hstack([P, P_rank])
    meta_T = np.hstack([T, T_rank])
    meta_oof = np.zeros(len(y))
    meta_splits = make_cv_splits(y, groups, RANDOM_WEIGHT_SEARCH_META_FOLDS, RANDOM_STATE+123)
    for fold, (tr_idx, va_idx) in enumerate(meta_splits):
        clf = LogisticRegression(C=0.5, solver='lbfgs', max_iter=1000, class_weight='balanced')
        clf.fit(meta_X[tr_idx], y[tr_idx])
        meta_oof[va_idx] = clf.predict_proba(meta_X[va_idx])[:, 1]
        print('stacker meta fold', fold, 'AUC', roc_auc_score(y[va_idx], meta_oof[va_idx]))
    final_clf = LogisticRegression(C=0.5, solver='lbfgs', max_iter=1000, class_weight='balanced')
    final_clf.fit(meta_X, y)
    meta_test = final_clf.predict_proba(meta_T)[:, 1]
    return meta_oof, meta_test

stack_models = top_models[:min(10, len(top_models))]
stack_oof, stack_test = logistic_stacker(stack_models)
save_blend('blend_logistic_stacker', stack_oof, stack_test)

blend_df = pd.DataFrame(blend_results).sort_values('oof_auc', ascending=False)
blend_df.to_csv(MODEL_DIR/'blend_results_v3.csv', index=False)
display(blend_df)

weight-search meta fold 0: valid AUC=0.650864, train_objective=0.660265


weight-search meta fold 1: valid AUC=0.661431, train_objective=0.660136


weight-search meta fold 2: valid AUC=0.660081, train_objective=0.660254


weight-search meta fold 3: valid AUC=0.672959, train_objective=0.656403


weight-search meta fold 4: valid AUC=0.654748, train_objective=0.662119


...

stacker meta fold 2 AUC 0.6462200234793959


stacker meta fold 3 AUC 0.6613178555688712


stacker meta fold 4 AUC 0.6699911474886442


saved submission_blend_logistic_stacker_v3.csv
blend_logistic_stacker: OOF AUC=0.659305


,blend,oof_auc,n_models
4,blend_cv_weight_search,0.660817,15
5,blend_logistic_stacker,0.659305,15
3,blend_greedy_oof,0.657268,15
1,blend_equal_rank_topk,0.653613,15
2,blend_auc_weighted_topk,0.653176,15
0,blend_equal_probability_topk,0.652828,15


In [9]:
# =========================
# 7. Final report
# =========================
results_df = pd.DataFrame(model_results).sort_values('oof_auc', ascending=False)
blend_df = pd.DataFrame(blend_results).sort_values('oof_auc', ascending=False)

print('Individual models:')
display(results_df)
print('Blends:')
display(blend_df)

all_report = pd.concat([
    results_df.rename(columns={'model': 'name'}).assign(kind='model'),
    blend_df.rename(columns={'blend': 'name'}).assign(kind='blend')
], ignore_index=True, sort=False).sort_values('oof_auc', ascending=False)
all_report.to_csv(MODEL_DIR/'all_results_v3.csv', index=False)

display(all_report.head(30))
print('Submissions saved in:', SUB_DIR)
print('Model outputs saved in:', MODEL_DIR)
print('Recommended first submissions:')
for name in all_report.head(8)['name'].tolist():
    print(' - submission_' + name + '_v3.csv')

Individual models:


,model,oof_auc,test_pred_mean,test_pred_std,elapsed_sec,mean_best_iter
17,flaml_automl,0.655824,0.015429,0.008104,6012.592753,NaN
16,catboost_depth4_ordered,0.648936,0.450481,0.112065,587.753351,NaN
2,lgb_small_leaves,0.647175,0.398239,0.134869,146.279542,405.8
0,lgb_regularized,0.643600,0.373386,0.134538,162.606043,360.6
7,logreg_saga_l1_sparse,0.641800,0.448931,0.154380,3206.059304,NaN
6,logreg_saga_l2,0.641793,0.448940,0.154350,1872.403902,NaN
15,catboost_depth6,0.641726,0.431195,0.118448,421.771670,NaN
3,lgb_goss,0.639295,0.320512,0.140195,191.239505,304.8
5,lgb_dart,0.638402,0.282832,0.128561,858.638413,1200.0
9,hist_gradient_boosting,0.632863,0.436696,0.107855,609.986326,NaN


Blends:


,blend,oof_auc,n_models
4,blend_cv_weight_search,0.660817,15
5,blend_logistic_stacker,0.659305,15
3,blend_greedy_oof,0.657268,15
1,blend_equal_rank_topk,0.653613,15
2,blend_auc_weighted_topk,0.653176,15
0,blend_equal_probability_topk,0.652828,15


,name,oof_auc,test_pred_mean,test_pred_std,elapsed_sec,mean_best_iter,kind,n_models
18,blend_cv_weight_search,0.660817,NaN,NaN,NaN,NaN,blend,15.0
19,blend_logistic_stacker,0.659305,NaN,NaN,NaN,NaN,blend,15.0
20,blend_greedy_oof,0.657268,NaN,NaN,NaN,NaN,blend,15.0
0,flaml_automl,0.655824,0.015429,0.008104,6012.592753,NaN,model,NaN
21,blend_equal_rank_topk,0.653613,NaN,NaN,NaN,NaN,blend,15.0
22,blend_auc_weighted_topk,0.653176,NaN,NaN,NaN,NaN,blend,15.0
23,blend_equal_probability_topk,0.652828,NaN,NaN,NaN,NaN,blend,15.0
1,catboost_depth4_ordered,0.648936,0.450481,0.112065,587.753351,NaN,model,NaN
2,lgb_small_leaves,0.647175,0.398239,0.134869,146.279542,405.8,model,NaN
3,lgb_regularized,0.643600,0.373386,0.134538,162.606043,360.6,model,NaN


Submissions saved in: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/submissions_v3
Model outputs saved in: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v3
Recommended first submissions:
 - submission_blend_cv_weight_search_v3.csv
 - submission_blend_logistic_stacker_v3.csv
 - submission_blend_greedy_oof_v3.csv
 - submission_flaml_automl_v3.csv
 - submission_blend_equal_rank_topk_v3.csv
 - submission_blend_auc_weighted_topk_v3.csv
 - submission_blend_equal_probability_topk_v3.csv
 - submission_catboost_depth4_ordered_v3.csv
